# 01 — Setup y extracción del informe FEMSA 2024

Prepara todo lo demás: monta Drive, clona el repo, instala dependencias con las versiones fijadas del proyecto, descarga el informe anual FEMSA 2024 (edición en inglés, 151 páginas) y extrae el texto por página + las tripletas por reglas. **No necesita Ollama** (la extracción es determinista, sin LLM).

**Orden:** corre este notebook después del 00 (o sin él: este notebook no usa Ollama, pero verifica que el servidor esté arriba para los que siguen).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Dependencias (versiones fijadas del proyecto)

Se instalan las mismas versiones que usaron las corridas originales (ver `repo/pyproject.toml` y los READMEs de `memo/`, `rag_baseline/` y `agentic_rag/`). En Colab, `torch` se instala sin el índice CPU: pip resuelve la rueda CUDA compatible con la T4. Si Colab ya trae un torch compatible, pip lo deja como está.

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 accelerate>=1.16.0 peft>=0.22.0 sentence-transformers==6.3.0 numpy>=2.5.5 pypdf>=7.1.1 neo4j>=6.4.1 rank-bm25==0.2.2 langchain==1.5.2 langgraph==1.2.24 langchain-core==1.8.5 langchain-huggingface==1.3.0 langchain-ollama

## Verificar Ollama (opcional aquí, obligatorio después)

Este notebook no usa Ollama, pero los notebooks 04–08 sí. Si el servidor no está arriba, corre el notebook `00_ollama_setup.ipynb` (o deja que cada notebook levante el suyo con su bloque idempotente).

In [ ]:
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
        print("Ollama responde en localhost:11434 (status", r.status, ")")
except Exception as e:
    print("Ollama NO responde:", e)
    print("Corre 00_ollama_setup.ipynb, o sigue: los notebooks 04-08 levantan el servidor solos.")

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive o generándolo del repo con `make extract` (`python pipeline/extract.py`).
3. **Solo si no lo subes**, generarlo aquí corriendo `extract.py` (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal.

`triples_seed.json` y `eval_questions.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive, o del repo con `make extract`.")
    print("Si no lo subes, el notebook lo genera con extract.py (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar con extract.py...")
        print("(descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        try:
            import pypdf
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1"], check=True)
        r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
        print("extract.py exit:", r.returncode)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

## Verificación: 151 páginas y la cifra clave

In [ ]:
import json

chunks = [json.loads(l) for l in open(REPO + "/pipeline/data/chunks.jsonl")]
print("paginas:", len(chunks), "| con texto:", sum(1 for c in chunks if c["chars"] > 0))
hits = [c["page"] for c in chunks if "781,585" in c["text"]]
print("781,585 aparece en paginas:", hits)
print("ejemplo p.11:", chunks[10]["text"][:300].replace("\n", " "))

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

El PDF (~19 MB) y `chunks.jsonl` quedan en Drive: son los insumos que los notebooks 02–08 pedirán primero (antes de subirlos o generarlos).

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('pipeline/data/chunks.jsonl', 'chunks.jsonl (151 páginas extraídas)', True), ('pipeline/data/FEMSA_2024_Annual_Report_EN.pdf', 'PDF del informe (inglés)', False), ('pipeline/extracted_triples.jsonl', '13 tripletas candidatas por reglas (pendientes de revisión)', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)